# Base model vs assistant

This notebook goes with the [Base model & post-training](https://bradleyboehmke.github.io/ai-engineering-notes/dictionary/base-model.html) dictionary entry. It sends one question to two models: a **base model** (SmolLM2-360M, pre-trained only) and **the same network after post-training** (SmolLM2-360M-Instruct). Then it:

1. **Shows what the assistant actually receives:** your question wrapped in its chat template.
2. **Compares first-token guesses:** where the two models split before either writes a word.
3. **Lets both write:** one answers and stops, and one keeps going.
4. **Gets an answer from the base model** by making the answer the natural next line (few-shot).
5. **Repeats it with GPT-2**, the base model from the earlier entries.

**No API key and no cost.** Everything runs on Colab's free CPU. The first run downloads about 1.4 GB of model weights, which takes a minute or two.

## 1. Check the libraries

Colab already has `transformers` and `torch`. If you run this somewhere else, `pip install transformers torch pandas matplotlib` first.

In [ ]:
import torch, transformers
print("transformers", transformers.__version__, "· torch", torch.__version__)

## 2. The code

This is the same code as [`base_vs_assistant.py`](https://github.com/bradleyboehmke/ai-engineering-notes/blob/main/learning/llm-mechanics/experiments/base_vs_assistant.py).

- `top5` runs the model once and returns its five most likely next tokens.
- `generate` is the [next-token prediction](https://bradleyboehmke.github.io/ai-engineering-notes/dictionary/next-token-prediction.html) loop written out by hand: pick the most likely token (greedy), append it, repeat. It stops on an end token or at `max_tokens`, and reports which one happened.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

def load(name):
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name).eval()
    return tok, model

def top5(tok, model, ids):
    """The model's five most likely next tokens, with probabilities."""
    with torch.no_grad():
        probs = model(ids).logits[0, -1].softmax(-1)
    top = torch.topk(probs, 5)
    return [(tok.decode(i), p.item()) for p, i in zip(top.values, top.indices)]

def generate(tok, model, ids, max_tokens):
    """Greedy loop: pick the top token, append it, repeat. Returns why it stopped."""
    end_tokens = {tok.eos_token_id, tok.convert_tokens_to_ids("<|im_end|>")}
    start = ids.shape[1]
    for _ in range(max_tokens):
        with torch.no_grad():
            next_id = model(ids).logits[0, -1].argmax().view(1, 1)
        ids = torch.cat([ids, next_id], dim=1)
        if next_id.item() in end_tokens:
            return tok.decode(ids[0, start:]), ids.shape[1] - start, "end token"
    return tok.decode(ids[0, start:]), max_tokens, "max_tokens"

def encode(tok, text):
    return tok(text, return_tensors="pt", add_special_tokens=False).input_ids

## 3. Load both models

Same architecture, same tokenizer, same size. The only difference is that the second one went through post-training.

In [ ]:
base_tok, base = load("HuggingFaceTB/SmolLM2-360M")
asst_tok, asst = load("HuggingFaceTB/SmolLM2-360M-Instruct")
print("Both loaded.")

## 4. What the assistant actually receives

Enter a question. A chat app or API doesn't send your bare text to an assistant. It wraps it in the model's **chat template** first: role markers, a default system prompt you didn't write, and an *opened* assistant turn for the model to fill.

The base model gets the bare question, because it was never trained on this format.

In [ ]:
prompt = "What is the capital of France?"  # @param {type:"string"}

chat = asst_tok.apply_chat_template(
    [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)

print(chat)
print(f"Bare question: {encode(base_tok, prompt).shape[1]} tokens · "
      f"with the template: {encode(asst_tok, chat).shape[1]} tokens")

## 5. First-token guesses

Before either model writes anything, look at what each one thinks comes next. **Before you run it**, predict: what is the base model's most likely next token after a bare question?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

guesses = {
    "Base model, bare question": top5(base_tok, base, encode(base_tok, prompt)),
    "Assistant, chat template": top5(asst_tok, asst, encode(asst_tok, chat)),
}

fig, axes = plt.subplots(1, 2, figsize=(9, 2.6), sharex=True)
for ax, (label, top), color in zip(axes, guesses.items(), ["#5b5a57", "#8a4620"]):
    names = [repr(t) for t, _ in top]
    ax.barh(names, [p for _, p in top], color=color)
    for i, (_, p) in enumerate(top):
        ax.text(p + 0.01, i, f"{p:.1%}", va="center", fontsize=9)
    ax.invert_yaxis()
    ax.set_title(label, loc="left", fontsize=10)
    ax.set_xlim(0, 1)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
fig.suptitle("Top 5 next tokens", x=0.01, ha="left", fontsize=11)
plt.tight_layout()
plt.show()

## 6. Let both write

Each model now writes up to `max_tokens` with the loop from step 2. **Before you run it**, predict: which one stops on its own, and roughly how many tokens does it use?

In [ ]:
max_tokens = 60  # @param {type:"integer"}

rows = []
for label, tok, model, text in [
    ("Base model, bare question", base_tok, base, prompt),
    ("Assistant, chat template", asst_tok, asst, chat),
]:
    out, n, reason = generate(tok, model, encode(tok, text), max_tokens)
    rows.append({"run": label, "tokens": n, "stopped by": reason, "output": out})

pd.set_option("display.max_colwidth", None)
pd.DataFrame(rows).set_index("run")

Read the base model's output with three questions:

- **Did it answer?** Is it responding to you, or writing the kind of page your question might appear on?
- **Did it stay on your question?** Or did it drift into other text that looks related?
- **Did it stop?** If `tokens` equals `max_tokens`, only the limit stopped it.

## 7. Get an answer from the base model (few-shot)

A base model answers when the text makes an answer the natural next line. Here the question is placed after two worked examples. **Before you run it**, predict: does it answer this time, and does it stop?

In [ ]:
few_shot = (
    "Q: What is the capital of Spain?\nA: Madrid\n\n"
    "Q: What is the capital of Japan?\nA: Tokyo\n\n"
    f"Q: {prompt}\nA:"
)
out, n, reason = generate(base_tok, base, encode(base_tok, few_shot), max_tokens)
print(few_shot + out)
print(f"\n--- {n} tokens, stopped by: {reason}")

It answers, then writes the next question itself, because in this kind of document another Q usually follows an A. You did the work an assistant does for you: you made the answer the likely continuation. You'd still need a stop sequence (for example `\n\n`) to cut it off.

## 8. The same with GPT-2

GPT-2 is the base model from the [Token](https://bradleyboehmke.github.io/ai-engineering-notes/dictionary/token.html) and [Next-token prediction](https://bradleyboehmke.github.io/ai-engineering-notes/dictionary/next-token-prediction.html) entries. It's older and smaller than SmolLM2, and it never had a post-trained version, so there's no assistant to compare it with. **Before you run it**, predict: does it behave more like the SmolLM2 base model or the assistant?

In [ ]:
gpt2_tok, gpt2 = load("gpt2")
ids = encode(gpt2_tok, prompt)
for t, p in top5(gpt2_tok, gpt2, ids):
    print(f"{p:6.1%}  {t!r}")
out, n, reason = generate(gpt2_tok, gpt2, ids, max_tokens)
print(f"\noutput: {out!r}\n--- {n} tokens, stopped by: {reason}")

## What to look for

- **The split happens at the first token.** The base model expects the document to go on, often with a newline. The assistant expects the start of a reply.
- **Only the assistant stops.** It chooses `<|im_end|>`, which closed every reply it saw in post-training. The base models run into `max_tokens`.
- **The base model "knows" the answer.** It wrote Paris, and with few-shot it answered cleanly. Post-training didn't add that knowledge. It changed what the model does with it.
- **The template is part of your input.** The assistant received more tokens than you typed, including a system prompt you didn't write.
- **Try your own question.** Change `prompt` in step 4 and rerun the cells below it. Try an instruction ("Write a haiku about rain") as well as a question.